In [ ]:
import glob
import os
import re

import dask.dataframe as dd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import date as dt_date
from matplotlib.patches import Patch
from pathlib import Path
from pyproj import Transformer

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]',
         '(1000, 5000]']

# First time

### Gauge data

In [ ]:
def create_duration_curve(values_series):
    """Create a duration curve from any values"""
    values = values_series.dropna().values
    values_sorted = np.sort(values)[::-1]  # Sort descending
    
    n = len(values_sorted)
    exceedance = np.arange(1, n + 1) / n * 100
    
    return exceedance, values_sorted

def year_fraction(dt):
    start = dt_date(dt.year, 1, 1).toordinal()
    year_length = dt_date(dt.year+1, 1, 1).toordinal() - start
    return round((dt.year + float(dt.toordinal() - start) / year_length), 1)

def calc_delta_h(lat):
    H_perm = -0.31460
    k_2 = 0.3
    lat_rad = np.radians(lat)
    delta_h = ((1 + k_2) * H_perm * np.sqrt(5/(4*np.pi)) * (((3/2) * np.sin(lat_rad)**2) - 0.5))
    return delta_h

def to_swot_crs(df):
    """Apply EPSG:6349 → EPSG:9057+3855 conversion (same as first script)"""
    df = df.copy()
    df['lon_wgs84'] = 0.0
    df['lat_wgs84'] = 0.0
    df['datum_offset_m'] = 0.0

    groups = df.groupby('epoch_01yr')
    total_groups = len(groups)
    
    print(f"Starting pyproj pipeline for {len(df)} points...")

    for i, (epoch_yr, group) in enumerate(groups, 1):
        epoch_val = float(epoch_yr)
        print(f"[{i}/{total_groups}] Epoch {epoch_val}: {len(group)} points", end="\r")

        transformer = Transformer.from_crs("EPSG:6349", "EPSG:9057+EPSG:3855", always_xy=True)
        epochs = [epoch_val] * len(group)
        
        new_lon, new_lat, alt_out, _ = transformer.transform(
            group['dec_long_va'].values,
            group['dec_lat_va'].values,
            group['alt_va'].values,
            epochs
        )

        df.loc[group.index, 'lon_wgs84'] = new_lon
        df.loc[group.index, 'lat_wgs84'] = new_lat
        df.loc[group.index, 'datum_offset_m'] = alt_out - group['alt_va'].values

    print(f"\nDone.")
    return df

In [ ]:
data_dir = Path('/nas/cee-water/cjgleason/fiona/USGS_gauges/sites_2026/')
parquet_files = sorted(data_dir.glob('*.parquet'))

gauge_metadata_path = '/nas/cee-water/cjgleason/fiona/USGS_gauges/sites_2026/../usgs_stations_with_stage_and_nhd_metadata.parquet'

# Create output directories
gauge_records_dir = 'gauge_records_by_site'
sdc_dir = 'stage_duration_curves_by_site'

os.makedirs(gauge_records_dir, exist_ok=True)
os.makedirs(sdc_dir, exist_ok=True)

gauge_metadata = pd.read_parquet(gauge_metadata_path)

# Find already processed gauges
processed_gauges = set(os.listdir(gauge_records_dir))
processed_gauges = {f.replace('.parquet', '') for f in processed_gauges}

print(f"Already processed: {len(processed_gauges)} gauges")

for idx, parquet_file in enumerate(parquet_files):
    gauge_id = parquet_file.stem
    
    # Skip if already processed
    if gauge_id in processed_gauges:
        continue
    
    try:
        df = pd.read_parquet(parquet_file)
        
        if 'stage' not in df.columns:
            continue
            
        df = df[df['stage'] != -999999.00]
        df['stage'] = df['stage'] * 0.3048
        
        meta = gauge_metadata[gauge_metadata['site_id'] == gauge_id]
        if meta.empty:
            continue
        
        df['dec_lat_va'] = meta['dec_lat_va'].values[0]
        df['dec_long_va'] = meta['dec_long_va'].values[0]
        df['alt_va'] = meta['alt_va'].values[0] * 0.3048
        df['epoch_01yr'] = df.index.to_series().apply(year_fraction)
        
        print(f"Converting CRS for {gauge_id}...")
        df = to_swot_crs(df)
        
        delta_h = calc_delta_h(df['dec_lat_va'].iloc[0])
        df['wse_gauge_swot_crs'] = df['stage'] + df['alt_va'] + df['datum_offset_m'] + delta_h
        df['site_id'] = gauge_id
        
        # Save gauge record immediately
        df.reset_index().to_parquet(os.path.join(gauge_records_dir, f'{gauge_id}.parquet'))
        
        # Create and save SDC immediately
        stage_series = df['wse_gauge_swot_crs']
        exceedance, stages = create_duration_curve(stage_series)
        
        sdc_df = pd.DataFrame({
            'site_id': gauge_id,
            'exceedance': exceedance,
            'stage': stages
        })
        sdc_df.to_parquet(os.path.join(sdc_dir, f'{gauge_id}.parquet'), index=False)
        
        print(f"✓ {gauge_id}: Saved ({len(stages)} SDC points)")
        
    except Exception as e:
        print(f"✗ {gauge_id}: Error - {e}")

print(f"\n✓ Processing complete!")
print(f"✓ Saved {len(os.listdir(gauge_records_dir))} gauge records")
print(f"✓ Saved {len(os.listdir(sdc_dir))} SDCs")

# Thereafter

In [ ]:
sdc_files = glob.glob('./stage_duration_curves_by_site/*.parquet')
sdc_data = {}

for sdc_file in sdc_files:
    df = pd.read_parquet(sdc_file)
    gauge_id = df['site_id'].iloc[0]
    sdc_data[gauge_id] = {
        'exceedance': df['exceedance'].values,
        'values': df['stage'].values,
        'site_id': gauge_id
    }

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]',
         '(1000, 5000]']

# Calculate variability metrics (same as before)
variability_metrics = []

for gauge_id, sdc in sdc_data.items():
    stages = sdc['values']
    exceedance = sdc['exceedance']
    
    gauge_rows = swot_quality_df[swot_quality_df['site_id'] == gauge_id]
    if len(gauge_rows) == 0:
        continue
    
    width_bin = gauge_rows['Bin'].iloc[0]
    width_m = gauge_rows['WidthM'].iloc[0]
    
    q25_idx = np.argmin(np.abs(exceedance - 75))
    q75_idx = np.argmin(np.abs(exceedance - 25))
    q50_idx = np.argmin(np.abs(exceedance - 50))
    
    q25_stage = stages[q25_idx]
    q75_stage = stages[q75_idx]
    q50_stage = stages[q50_idx]
    
    metrics = {
        'site_id': gauge_id,
        'Bin': width_bin,
        'WidthM': width_m,
        'stage_range': max(0, stages.max() - stages.min()),  # ← BOUND AT 0
        'stage_iqr': max(0, q75_stage - q25_stage),           # ← BOUND AT 0
        'stage_mean': stages.mean(),
        'stage_median': q50_stage,
        'stage_std': max(0, stages.std()),                    # ← BOUND AT 0
        'cv': max(0, stages.std() / stages.mean()) if stages.mean() != 0 else np.nan,  # ← BOUND AT 0
        'stage_min': stages.min(),
        'stage_max': stages.max(),
        'n_observations': len(stages),
    }
    
    variability_metrics.append(metrics)

# Create dataframe
variability_df = pd.DataFrame(variability_metrics)

# Convert Bin to categorical with your order
variability_df['Bin'] = pd.Categorical(variability_df['Bin'], categories=order, ordered=True)
variability_df = variability_df.sort_values('Bin')

# Summary by width bin
summary_by_bin = variability_df.groupby('Bin', observed=False).agg({
    'stage_range': ['mean', 'std', 'min', 'max'],
    'stage_iqr': ['mean', 'std'],
    'cv': ['mean', 'std'],
    'stage_mean': 'mean',
    'WidthM': 'mean',
    'site_id': 'count'
}).round(3)

summary_by_bin.columns = ['_'.join(col).strip() for col in summary_by_bin.columns.values]
summary_by_bin = summary_by_bin.rename(columns={'site_id_count': 'n_gauges'})

# ===== PLOTTING =====
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Range by bin
ax = axes[0, 0]
bin_means = variability_df.groupby('Bin', observed=False)['stage_range'].mean()
bin_std = variability_df.groupby('Bin', observed=False)['stage_range'].std()
ax.bar(range(len(bin_means)), bin_means.values, yerr=bin_std.values, capsize=5, alpha=0.7, color='steelblue')
ax.set_xticks(range(len(bin_means)))
ax.set_xticklabels(bin_means.index, rotation=45, ha='right')
ax.set_ylabel('Stage Range (m)')
ax.set_title('Flow Variability (Range) by Width Bin')
ax.set_ylim(bottom=0)  # ← BOUND Y-AXIS AT 0
ax.grid(axis='y', alpha=0.3)

# Plot 2: CV by bin
ax = axes[0, 1]
bin_means = variability_df.groupby('Bin', observed=False)['cv'].mean()
bin_std = variability_df.groupby('Bin', observed=False)['cv'].std()
ax.bar(range(len(bin_means)), bin_means.values, yerr=bin_std.values, capsize=5, alpha=0.7, color='coral')
ax.set_xticks(range(len(bin_means)))
ax.set_xticklabels(bin_means.index, rotation=45, ha='right')
ax.set_ylabel('Coefficient of Variation')
ax.set_title('Normalized Variability (CV) by Width Bin')
ax.set_ylim(bottom=0)  # ← BOUND Y-AXIS AT 0
ax.grid(axis='y', alpha=0.3)

# Plot 3: IQR by bin
ax = axes[1, 0]
bin_means = variability_df.groupby('Bin', observed=False)['stage_iqr'].mean()
bin_std = variability_df.groupby('Bin', observed=False)['stage_iqr'].std()
ax.bar(range(len(bin_means)), bin_means.values, yerr=bin_std.values, capsize=5, alpha=0.7, color='seagreen')
ax.set_xticks(range(len(bin_means)))
ax.set_xticklabels(bin_means.index, rotation=45, ha='right')
ax.set_ylabel('Stage IQR (m)')
ax.set_title('Interquartile Range by Width Bin')
ax.set_ylim(bottom=0)  # ← BOUND Y-AXIS AT 0
ax.grid(axis='y', alpha=0.3)

# Plot 4: Number of gauges per bin
ax = axes[1, 1]
n_gauges = variability_df.groupby('Bin', observed=False)['site_id'].nunique()
ax.bar(range(len(n_gauges)), n_gauges.values, alpha=0.7, color='mediumpurple')
ax.set_xticks(range(len(n_gauges)))
ax.set_xticklabels(n_gauges.index, rotation=45, ha='right')
ax.set_ylabel('Number of Gauges')
ax.set_title('Sample Size by Width Bin')
ax.set_ylim(bottom=0)  # ← BOUND Y-AXIS AT 0
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
# plt.savefig(os.path.join(save_dir, 'flow_variability_by_width_bin.png'), dpi=300, bbox_inches='tight')
# print(f"Plot saved to {save_dir}/flow_variability_by_width_bin.png")
plt.show()

In [ ]:
# Plot each of the first 10 as individual full-size plots
for gauge_id in gauge_ids[0:10]:
    data = sdc_data[gauge_id]
    plt.figure(figsize=(10, 6))
    plt.semilogy(data['exceedance'], data['values'], 'b-', linewidth=2.5)
    plt.xlabel('Exceedance Probability (%)', fontsize=12)
    plt.ylabel('Stage [m]', fontsize=12)
    plt.title(f'Stage Duration Curve - Gauge {gauge_id}', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 100)
    plt.tight_layout()
    plt.show()
    # plt.savefig(f'fdc_{gauge_id}.png', dpi=300, bbox_inches='tight')
    # plt.close()
    # print(f"Saved: fdc_{gauge_id}.png")

In [ ]:
results_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = './figures_stage_duration_curve/'

# Load your saved quality data
swot_quality_df = pd.read_parquet(os.path.join(results_dir, 'swot_observations_with_quality.parquet'))

# Debug: Check what's in your SDC data
for bin_val in order[:-3]:
    bin_data = swot_quality_df[swot_quality_df['Bin'] == bin_val]
    gauge_obs_counts = bin_data['site_id'].value_counts()
    selected_gauges = gauge_obs_counts[gauge_obs_counts >= 20].index.tolist()
    
    if selected_gauges:
        test_gauge = selected_gauges[0]
        if test_gauge in sdc_data:
            data = sdc_data[test_gauge]
            print(f"\nBin {bin_val}, Gauge {test_gauge}:")
            print(f"  Stage values (first 10): {data['values'][:10]}")
            print(f"  Stage min: {np.min(data['values'])}, max: {np.max(data['values'])}")
            print(f"  Exceedance min: {np.min(data['exceedance'])}, max: {np.max(data['exceedance'])}")

In [ ]:
results_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = './figures_stage_duration_curve/'

# Load your saved quality data
swot_quality_df = pd.read_parquet(os.path.join(results_dir, 'swot_observations_with_quality.parquet'))

# Color mapping
# quality_colors = {
#     'unfiltered': 'silver',
#     'useable': sns.color_palette('tab20b')[2],
#     'height_threshold': '#8DAE91',
#     'high_quality': sns.color_palette('tab20b')[-3]
# }

quality_colors = {
    'unfiltered':       'gainsboro',    # light gray, insufficient data
    'useable':          'tab:blue',     # unacceptable bias
    'height_threshold': 'tab:orange',   # unreasonable timeseries
    'high_quality':     'tab:green'     # sensible SWOT WSE
}

legend_labels = ['Insufficient data (pixels)',
                 'Unacceptable bias',
                 'Unreasonable timeseries',
                 'Sensible SWOT WSE']

# Create one figure per bin
for bin_val in order[:-3]:
    # Create safe filename from bin value
    safe_bin_name = re.sub(r'[^\w\-_]', '_', str(bin_val).replace(' ', '_'))
    
    # Get all gauges in this bin
    bin_data = swot_quality_df[swot_quality_df['Bin'] == bin_val]
    bin_gauges = bin_data['site_id'].unique()
    
    # Filter for reaches with >= 20 observations
    gauge_obs_counts = bin_data['site_id'].value_counts()
    selected_gauges = gauge_obs_counts[gauge_obs_counts >= 20].index.tolist()
    
    print(f"Width Bin {bin_val}: {len(selected_gauges)} reaches with >= 20 observations")
    
    # Split gauges into chunks of 20
    chunk_size = 10
    gauge_chunks = [selected_gauges[i:i+chunk_size] for i in range(0, len(selected_gauges), chunk_size)]
    
    # Create a figure for each chunk
    for chunk_idx, gauge_chunk in enumerate(gauge_chunks):
        fig, ax = plt.subplots(figsize=(12, 12))
        
        # Plot each selected gauge in this chunk
        for gauge_id in gauge_chunk:
            if gauge_id in sdc_data:
                # Get width for this gauge
                gauge_width = swot_quality_df[swot_quality_df['site_id'] == gauge_id]['WidthM'].iloc[0]
                
                # Plot the SDC (normalize PER GAUGE)
                data = sdc_data[gauge_id]
                stage_min = np.min(data['values'])
                stage_max = np.max(data['values'])
                stage_range = stage_max - stage_min
                
                # Normalize this gauge's stages to 0-1
                normalized_stages = (data['values'] - stage_min) / stage_range if stage_range > 0 else data['values']
                
                ax.plot(data['exceedance'], normalized_stages, linewidth=2,
                        label=f'{gauge_id} ({gauge_width:.1f}m)')
                
                # Plot observations for this gauge
                gauge_obs = swot_quality_df[swot_quality_df['site_id'] == gauge_id]
                
                if len(gauge_obs) > 0:
                    all_gauge_wse = gauge_obs['wse_gauge'].dropna().values
                    
                    for quality_level in ['unfiltered', 'useable', 'height_threshold', 'high_quality']:
                        obs = gauge_obs[gauge_obs['quality_level'] == quality_level]
                        
                        if len(obs) > 0:
                            for _, row in obs.iterrows():
                                if pd.notna(row['wse_gauge']):
                                    percentile = (all_gauge_wse <= row['wse_gauge']).sum() / len(all_gauge_wse) * 100
                                    exceedance = 100 - percentile
                                    
                                    all_stages = data['values']
                                    idx = np.argmin(np.abs(data['exceedance'] - exceedance))
                                    stage_value = all_stages[idx]
                                    
                                    # Normalize this point using the same gauge's min/max
                                    normalized_stage_point = (stage_value - stage_min) / stage_range if stage_range > 0 else stage_value
                                    
                                    ax.scatter(
                                        exceedance,
                                        normalized_stage_point,
                                        color=quality_colors[quality_level],
                                        s=40,
                                        edgecolors='k',
                                        linewidth=1,
                                        zorder=5
                                    )
        
        # Set up axes labels and title
        ax.set_xlabel('Exceedance Probability (%)', fontsize=14)
        ax.set_ylabel('Normalized Stage (0-1)', fontsize=14)
        
        # Add chunk info to title if there are multiple chunks
        if len(gauge_chunks) > 1:
            ax.set_title(f'Width Bin: {bin_val} (Chunk {chunk_idx+1}/{len(gauge_chunks)})', 
                        fontsize=14, fontweight='bold')
        else:
            ax.set_title(f'Width Bin: {bin_val}', fontsize=14, fontweight='bold')
        
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-3, 103)
        ax.set_ylim(-0.05, 1.05)
        
        # Add legend for quality levels
        from matplotlib.patches import Patch
        quality_levels = ['unfiltered', 'useable', 'height_threshold', 'high_quality']
        legend_elements = [
            Patch(facecolor=quality_colors[level], edgecolor='k', label=label)
            for level, label in zip(quality_levels, legend_labels)
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=10, title='Quality Level')
        
        plt.tight_layout()
        
        # Save with chunk number if multiple chunks
        if len(gauge_chunks) > 1:
            plt.savefig(os.path.join(save_dir, f'sdc_bin_{safe_bin_name}_chunk_{chunk_idx+1}_of_{len(gauge_chunks)}.png'), 
                        dpi=300, bbox_inches='tight')
            print(f"Saved: sdc_bin_{safe_bin_name}_chunk_{chunk_idx+1}_of_{len(gauge_chunks)}.png")
        else:
            plt.savefig(os.path.join(save_dir, f'sdc_bin_{safe_bin_name}.png'), 
                        dpi=300, bbox_inches='tight')
            print(f"Saved: sdc_bin_{safe_bin_name}.png")
        
        plt.show()

print("\n✓ All plots complete!")

In [ ]:
# Check the ranges
print("SDC stage values (meters):")
for gauge_id in list(sdc_data.keys())[:3]:
    vals = sdc_data[gauge_id]['values']
    print(f"  {gauge_id}: {vals.min():.2f} to {vals.max():.2f}")

print("\nSWOT wse_gauge values (???):")
for gauge_id in list(sdc_data.keys())[:3]:
    obs = swot_quality_df[swot_quality_df['site_id'] == gauge_id]
    if len(obs) > 0:
        vals = obs['wse_gauge'].dropna()
        print(f"  {gauge_id}: {vals.min():.2f} to {vals.max():.2f}")